# Comparación con NLP: spaCy y Transformers

**Responsable:** Persona 1

Se procesa el conjunto de prueba (`data/test_set.csv`) con dos herramientas de NLP para español y se
guardan las salidas en el formato acordado (`docs/gramatica_es.md`, sección 2.4):

| Herramienta | Salida |
| --- | --- |
| spaCy `es_core_news_sm` | `results/spacy.jsonl` |
| Modelo de HuggingFace Transformers para español | `results/transformers.jsonl` |

Ninguna de las dos decide si la oración es válida (`acepta` = `null`): se comparan por **etiquetado**
y, spaCy, por **funciones sintácticas** (sujeto, verbo y objeto). La conversión a nuestras categorías es
la de la sección 2.5, implementada en `experiments/tagsets.py`, para que nadie la invente por su cuenta.

**Antes de correrlo:**

```bash
pip install -r requirements-experiments.txt
python -m spacy download es_core_news_sm
```

Preguntas que este notebook debe responder (van al informe):

1. ¿En qué oraciones spaCy y Transformers analizan bien lo que la CFG **no puede**? Pista: `en_gramatica = no` y `espanol_valido = si`.
2. ¿En qué oraciones la CFG acierta y ellos **no**?

In [ ]:
# Configuración: el notebook vive en notebooks/, el código del proyecto en la raíz.
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from evaluation.formats import Prediction, load_test_set, save_predictions, results_path, load_predictions

items = load_test_set()
print(f"{len(items)} oraciones en el conjunto de prueba")

In [ ]:
for item in items:
    marca = "✓" if item.in_grammar else "✗"
    print(f"{item.id:>3}  nivel {item.level}  {marca}  {item.sentence}")

## 1. spaCy

In [ ]:
import spacy

try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    raise SystemExit("Falta el modelo: python -m spacy download es_core_news_sm")

print(spacy.__version__, nlp.meta["name"], nlp.meta["version"])   # anotar en el informe

In [ ]:
from experiments.tagsets import spacy_output

spacy_predictions = []
for item in items:
    start = time.perf_counter()
    try:
        doc = nlp(item.sentence)
        row = spacy_output(item.id, doc, (time.perf_counter() - start) * 1000)
        spacy_predictions.append(Prediction.from_json(row))
    except Exception as e:  # el formato pide escribir la línea aunque falle
        spacy_predictions.append(Prediction(item.id, "spacy", accepts=None,
                                            run_error=f"{type(e).__name__}: {e}"))

print("Guardado en", save_predictions("spacy", spacy_predictions, items))

In [ ]:
# Vista rápida: etiquetas de spaCy frente a las esperadas.
for item, pred in zip(items, spacy_predictions):
    expected = [c for _, c in item.tags]
    got = [c for _, c in (pred.tags or [])]
    mark = "OK " if got == expected else "DIF"
    print(f"{mark} {item.id:>3} {item.sentence}")
    if got != expected:
        print("      esperado:", expected)
        print("      spaCy:   ", got)

## 2. HuggingFace Transformers

TODO(P1): elegir el modelo. Tiene que ser de **clasificación de tokens** (POS tagging) para español
con etiquetas UPOS. Buscarlo en huggingface.co (filtro *token-classification*, idioma *es*) y anotar
en el informe el nombre exacto, el tamaño y el tagset.

Si el modelo solo hace POS, `sujeto`, `verbo` y `objeto` quedan en `null` (sección 2.4).

In [ ]:
from transformers import pipeline

MODEL = "TODO(P1): nombre/del-modelo"

tagger = pipeline("token-classification", model=MODEL, aggregation_strategy="simple")

In [ ]:
from experiments.tagsets import upos_category


def transformers_tags(sentence: str) -> list[list[str]]:
    """[[palabra, categoría], ...] con la conversión de la sección 2.5 (sin rasgos).

    TODO(P1): revisar con salidas reales que cada entidad sea UNA palabra
    (aggregation_strategy) y que la etiqueta sea UPOS; si no, adaptar aquí.
    """
    tags = []
    for entity in tagger(sentence):
        word = entity["word"].strip().lower()
        category = upos_category(entity["entity_group"], word)
        if category is not None:
            tags.append([word, category])
    return tags


transformers_predictions = []
for item in items:
    start = time.perf_counter()
    try:
        tags = transformers_tags(item.sentence)
        transformers_predictions.append(Prediction(
            item.id, "transformers", accepts=None, tags=tags,
            time_ms=(time.perf_counter() - start) * 1000))
    except Exception as e:
        transformers_predictions.append(Prediction(item.id, "transformers", accepts=None,
                                                   run_error=f"{type(e).__name__}: {e}"))

print("Guardado en", save_predictions("transformers", transformers_predictions, items))

## 3. Comparación con la CFG

Necesita `results/parser.jsonl` (Persona 2: `python -m evaluation.run_parser`).

In [ ]:
systems = [s for s in ("parser", "spacy", "transformers") if results_path(s).exists()]
by_system = {s: {p.id: p for p in load_predictions(s)} for s in systems}

def tags_ok(item, pred):
    return pred is not None and pred.tags is not None and [c for _, c in pred.tags] == [c for _, c in item.tags]

print(f"{'id':>3} {'en_gram':<7} {'esp':<4} " + " ".join(f"{s:<13}" for s in systems))
for item in items:
    cells = []
    for s in systems:
        p = by_system[s].get(item.id)
        decision = "—" if p is None or p.accepts is None else ("acepta" if p.accepts else "rechaza")
        cells.append(f"{decision}/{'tags✓' if tags_ok(item, p) else 'tags✗'}")
    print(f"{item.id:>3} {str(item.in_grammar):<7} {str(item.valid_spanish):<4} " + " ".join(f"{c:<13}" for c in cells))

### Conclusiones (para el informe)

TODO(P1):

- Oraciones que spaCy/Transformers analizan bien y la CFG no puede (sujeto omitido, orden libre, subordinadas, palabras fuera del léxico…).
- Oraciones que la CFG acierta y ellos no (por ejemplo, las de concordancia: un etiquetador no las rechaza).
- Diferencia de fondo: reglas explícitas frente a modelos estadísticos.